# Chapter 2 · Training and Saving Neural Networks

Steps: 1) download `UCI_Credit_Card.csv` (see README) · 2) on Kaggle: Add Input → Upload the CSV, turn on GPU · 3) Run All.

Cells marked *pre-run* take longer; run them before class or wait for them.

**D2-01 · Setup: imports, seed, versions**

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import glob, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf, keras
keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()
print("TF", tf.__version__, "| Keras", keras.__version__)

**D2-02 · Load the data (same code as D1-14)**

In [ ]:
path = glob.glob("/kaggle/input/**/UCI_Credit_Card.csv", recursive=True) + glob.glob("UCI_Credit_Card.csv")
df = pd.read_csv(path[0])
cols = df.columns.drop(["ID", "default.payment.next.month"])
X = df[cols].to_numpy("float32")
y = df["default.payment.next.month"].to_numpy()
print(X.shape, "| default rate:", round(y.mean(), 4))

**D2-03 · Split FIRST: train / validation / test (stratified)**

In [ ]:
from sklearn.model_selection import train_test_split
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.2, stratify=y_tmp, random_state=42)
for name, part in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s} rows={len(part):6d}  default rate={part.mean():.4f}")

**D2-04 · Scale SECOND: mean and sd from X_train only**

In [ ]:
mu, sd = X_train.mean(axis=0), X_train.std(axis=0)
Xs_train, Xs_val, Xs_test = (X_train - mu) / sd, (X_val - mu) / sd, (X_test - mu) / sd
print("train mean/sd:", Xs_train.mean().round(3), Xs_train.std().round(3))
print("val   mean/sd:", Xs_val.mean().round(3), Xs_val.std().round(3))
leaky_mu = X.mean(axis=0)   # D1-16 style: all 30,000 rows, test included
print("LIMIT_BAL mean  train-only:", round(mu[0]), "| all rows:", round(leaky_mu[0]))

**D2-05 · The untrained 23-16-2 network (D1-16) on validation clients**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
def report(model, Xs, y_true):
    pred = model.predict(Xs, verbose=0).argmax(axis=1)
    return {"accuracy": round(accuracy_score(y_true, pred), 4),
            "precision": round(precision_score(y_true, pred, zero_division=0), 4),
            "recall": round(recall_score(y_true, pred), 4)}
credit_net = keras.Sequential([keras.Input(shape=(23,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(2, activation="softmax")])
print("Parameters:", credit_net.count_params())
before = report(credit_net, Xs_val, y_val)
print("Untrained on validation:", before)
print("Baseline 'always no default':", round(1 - y_val.mean(), 4))

**D2-06 · Loss functions: BCE vs MSE for a real defaulter (y = 1)**

In [ ]:
p = np.array([0.9, 0.5, 0.1, 0.01])        # predicted P(default)
bce = -np.log(p)                             # y = 1, so BCE = -ln(p)
mse = (1 - p) ** 2
print(pd.DataFrame({"p": p, "BCE": bce.round(4), "MSE": mse.round(4)}))

**D2-07 · Two outputs + softmax: sparse categorical CE = BCE**

In [ ]:
probs = np.array([[0.3, 0.7]])               # [P(no default), P(default)]
scce = keras.losses.SparseCategoricalCrossentropy()(np.array([1]), probs)
bce1 = keras.losses.BinaryCrossentropy()(np.array([[1.0]]), np.array([[0.7]]))
print("sparse CE:", round(float(scce), 4), "| BCE:", round(float(bce1), 4))
cce3 = keras.losses.CategoricalCrossentropy()(np.array([[0, 0, 1.0]]), np.array([[0.2, 0.3, 0.5]]))
print("3-class categorical CE:", round(float(cce3), 4))

**D2-08 · Backprop by hand: one neuron, chain rule**

In [ ]:
x, y1, w, b, lr = 2.0, 1.0, 0.5, -1.0, 0.1
z = w * x + b; a = 1 / (1 + np.exp(-z)); L = -np.log(a)
dL_dz = a - y1                     # sigmoid + BCE gives this simple form
dL_dw, dL_db = dL_dz * x, dL_dz * 1
print(f"z={z:.2f} a={a:.4f} L={L:.4f} | dL/dw={dL_dw:.4f} dL/db={dL_db:.4f}")

**D2-09 · Same step with tf.GradientTape: gradients, update, loss again**

In [ ]:
w_t, b_t = tf.Variable(0.5), tf.Variable(-1.0)
def loss_fn():
    return -tf.math.log(tf.sigmoid(w_t * x + b_t))
with tf.GradientTape() as tape:
    L0 = loss_fn()
gw, gb = tape.gradient(L0, [w_t, b_t])
w_t.assign_sub(lr * gw); b_t.assign_sub(lr * gb)
print(f"grads: {gw.numpy():.4f} {gb.numpy():.4f} | new w={w_t.numpy():.2f} b={b_t.numpy():.2f}")
print(f"loss before={L0.numpy():.4f}  after={loss_fn().numpy():.4f}")

**D2-10 · Train the 23-16-2 network: 2 epochs (live)**

In [ ]:
credit_net.compile(optimizer=keras.optimizers.Adam(0.001),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h = credit_net.fit(Xs_train, y_train, epochs=2, batch_size=64,
                   validation_data=(Xs_val, y_val), verbose=2)
print("steps per epoch:", int(np.ceil(len(Xs_train) / 64)))

**D2-11 · Continue to 20 epochs and plot the loss curve (pre-run)**

In [ ]:
h2 = credit_net.fit(Xs_train, y_train, epochs=20, initial_epoch=2, batch_size=64,
                    validation_data=(Xs_val, y_val), verbose=0)
hist = {k: h.history[k] + h2.history[k] for k in h.history}
json.dump(hist, open("hist_credit.json", "w"))
plt.plot(range(1, 21), hist["loss"], label="train"); plt.plot(range(1, 21), hist["val_loss"], label="validation")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("23-16-2 credit_net"); plt.savefig("fig_credit_loss.png", dpi=150); plt.show()
print("val loss epoch 1 / 10 / 20:", [round(hist["val_loss"][i], 4) for i in (0, 9, 19)])

**D2-12 · Four optimizers, same start weights (pre-run)**

In [ ]:
def build_net(init="glorot_uniform"):
    return keras.Sequential([keras.Input(shape=(23,)),
        keras.layers.Dense(16, activation="relu", kernel_initializer=init),
        keras.layers.Dense(2, activation="softmax")])
keras.utils.set_random_seed(42); w0 = build_net().get_weights()
opts = {"SGD": keras.optimizers.SGD(0.01), "Momentum": keras.optimizers.SGD(0.01, momentum=0.9),
        "RMSprop": keras.optimizers.RMSprop(0.001), "Adam": keras.optimizers.Adam(0.001)}
opt_hist = {}
for name, opt in opts.items():
    keras.utils.set_random_seed(42); m = build_net(); m.set_weights(w0)
    m.compile(optimizer=opt, loss="sparse_categorical_crossentropy")
    opt_hist[name] = m.fit(Xs_train, y_train, epochs=15, batch_size=64, validation_data=(Xs_val, y_val), verbose=0).history
json.dump(opt_hist, open("hist_optimizers.json", "w"))

**D2-13 · Compare the four loss curves**

In [ ]:
opt_hist = json.load(open("hist_optimizers.json"))
for name, hh in opt_hist.items():
    plt.plot(range(1, 16), hh["loss"], label=name)
plt.xlabel("epoch"); plt.ylabel("training loss"); plt.legend(); plt.title("Same model, same start, four optimizers")
plt.savefig("fig_optimizers.png", dpi=150); plt.show()
for name, hh in opt_hist.items():
    print(f"{name:9s} train loss ep1={hh['loss'][0]:.4f} ep15={hh['loss'][-1]:.4f} | val ep15={hh['val_loss'][-1]:.4f}")

**D2-14 · Vanishing gradients: 10 sigmoid layers vs 10 ReLU layers**

In [ ]:
def deep_net(act, init="glorot_uniform", bn=False):
    layers = [keras.Input(shape=(23,))]
    for _ in range(10):
        layers.append(keras.layers.Dense(16, activation=act, kernel_initializer=init))
        if bn: layers.append(keras.layers.BatchNormalization())
    layers.append(keras.layers.Dense(2, activation="softmax"))
    return keras.Sequential(layers)
xb, yb = Xs_train[:256], y_train[:256]
loss_obj = keras.losses.SparseCategoricalCrossentropy()
def grad_norms(model):
    with tf.GradientTape() as tape:
        L = loss_obj(yb, model(xb))
    grads = tape.gradient(L, model.trainable_weights)
    return [float(tf.norm(g)) for g, v in zip(grads, model.trainable_weights) if "kernel" in v.path][:10]
keras.utils.set_random_seed(42); g_sig = grad_norms(deep_net("sigmoid"))
keras.utils.set_random_seed(42); g_relu = grad_norms(deep_net("relu"))
print(pd.DataFrame({"layer": range(1, 11), "sigmoid": np.round(g_sig, 6), "relu": np.round(g_relu, 6)}).to_string(index=False))

**D2-15 · Exploding gradients: weights started too large**

In [ ]:
big = keras.initializers.RandomNormal(stddev=1.0)
keras.utils.set_random_seed(42); g_big = grad_norms(deep_net("relu", init=big))
print(pd.DataFrame({"layer": range(1, 11), "relu, stddev=1.0": np.round(g_big, 2)}).to_string(index=False))

**D2-16 · Initialization: Glorot vs He in a deep ReLU network**

In [ ]:
def act_stds(model):
    a, out = xb, []
    for layer in model.layers[:10]:
        a = layer(a); out.append(float(tf.math.reduce_std(a)))
    return out
keras.utils.set_random_seed(42); s_glorot = act_stds(deep_net("relu", "glorot_uniform"))
keras.utils.set_random_seed(42); s_he = act_stds(deep_net("relu", "he_normal"))
print(pd.DataFrame({"layer": range(1, 11), "Glorot": np.round(s_glorot, 4), "He": np.round(s_he, 4)}).to_string(index=False))

**D2-17 · Batch Normalization: deep sigmoid network with vs without BN (pre-run)**

In [ ]:
bn_hist = {}
for name, use_bn in [("no BN", False), ("with BN", True)]:
    keras.utils.set_random_seed(42); m = deep_net("sigmoid", bn=use_bn)
    m.compile(optimizer=keras.optimizers.SGD(0.01), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    bn_hist[name] = m.fit(Xs_train, y_train, epochs=15, batch_size=64, validation_data=(Xs_val, y_val), verbose=0).history
json.dump(bn_hist, open("hist_bn.json", "w"))
for name, hh in bn_hist.items():
    plt.plot(range(1, 16), hh["val_loss"], label=name)
q = y_train.mean(); plt.axhline(-(q*np.log(q) + (1-q)*np.log(1-q)), ls=":", c="gray", label="no learning")
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.legend(); plt.title("10-layer sigmoid network")
plt.savefig("fig_bn.png", dpi=150); plt.show()
for name, hh in bn_hist.items():
    print(f"{name:8s} val loss ep1={hh['val_loss'][0]:.4f} ep15={hh['val_loss'][-1]:.4f} | val acc ep15={hh['val_accuracy'][-1]:.4f}")

**D2-18 · Overfitting: a wide network with no regularization (pre-run)**

In [ ]:
def wide_net(l1=0.0, l2=0.0, drop=0.0):
    reg = keras.regularizers.L1L2(l1=l1, l2=l2)
    layers = [keras.Input(shape=(23,))]
    for _ in range(3):
        layers.append(keras.layers.Dense(256, activation="relu", kernel_initializer="he_normal", kernel_regularizer=reg))
        if drop: layers.append(keras.layers.Dropout(drop))
    layers.append(keras.layers.Dense(2, activation="softmax"))
    return keras.Sequential(layers)
def run_wide(**kw):
    keras.utils.set_random_seed(42); m = wide_net(**kw)
    m.compile(optimizer=keras.optimizers.Adam(0.001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m.fit(Xs_train, y_train, epochs=30, batch_size=128, validation_data=(Xs_val, y_val), verbose=0).history
reg_hist = {"none": run_wide()}
hh = reg_hist["none"]
plt.plot(range(1, 31), hh["loss"], label="train"); plt.plot(range(1, 31), hh["val_loss"], label="validation")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("256-256-256, no regularization")
plt.savefig("fig_overfit.png", dpi=150); plt.show()
print("best val loss at epoch", int(np.argmin(hh["val_loss"])) + 1, "| final train", round(hh["loss"][-1], 4), "val", round(hh["val_loss"][-1], 4))

**D2-19 · Same network + L2 (and L1) (pre-run)**

In [ ]:
reg_hist["L2 0.001"] = run_wide(l2=0.001)
reg_hist["L1 0.0001"] = run_wide(l1=0.0001)
for name in ["L2 0.001", "L1 0.0001"]:
    print(f"{name:9s} final train={reg_hist[name]['loss'][-1]:.4f} val={reg_hist[name]['val_loss'][-1]:.4f}")

**D2-20 · Same network + Dropout, and the gap table (pre-run)**

In [ ]:
reg_hist["Dropout 0.3"] = run_wide(drop=0.3)
json.dump(reg_hist, open("hist_reg.json", "w"))
ce = keras.losses.SparseCategoricalCrossentropy()
rows = []
for name, hh in reg_hist.items():
    rows.append({"model": name, "train acc": round(hh["accuracy"][-1], 4), "val acc": round(hh["val_accuracy"][-1], 4),
                 "acc gap": round(hh["accuracy"][-1] - hh["val_accuracy"][-1], 4), "val loss": round(hh["val_loss"][-1], 4)})
print(pd.DataFrame(rows).to_string(index=False))

**D2-21 · Our trained 23-16-2 network on validation: before vs after**

In [ ]:
after = report(credit_net, Xs_val, y_val)
print("After 20 epochs, validation:", after)
print(pd.DataFrame({"untrained (D2-05)": before, "trained": after}))

**D2-22 · The test set, used ONCE**

In [ ]:
test_res = report(credit_net, Xs_test, y_test)
print("TEST:", test_res)
print("Baseline 'always no default' on test:", round(1 - y_test.mean(), 4))
print("Defaulters in test:", int(y_test.sum()), "| caught:", int(round(test_res["recall"] * y_test.sum())))

**D2-23 · Save the model + scaler, reload, check identical predictions**

In [ ]:
credit_net.save("credit_mlp.keras")
json.dump({"columns": list(cols), "mean": mu.tolist(), "sd": sd.tolist()}, open("scaler.json", "w"))
reloaded = keras.models.load_model("credit_mlp.keras")
sc = json.load(open("scaler.json"))
Xs2 = (X_test - np.array(sc["mean"], "float32")) / np.array(sc["sd"], "float32")
print("Identical predictions:", np.allclose(credit_net.predict(Xs_test, verbose=0), reloaded.predict(Xs2, verbose=0)))
print("File sizes (KB):", round(os.path.getsize("credit_mlp.keras") / 1024, 1), round(os.path.getsize("scaler.json") / 1024, 1))

**D2-24 · One known client from the test set, for the API check**

In [ ]:
i = 0
client = {c: float(v) for c, v in zip(cols, X_test[i])}
p_default = float(reloaded.predict(Xs2[i:i + 1], verbose=0)[0, 1])
json.dump({"features": client, "true_label": int(y_test[i]), "p_default": p_default}, open("known_client.json", "w"))
print({k: client[k] for k in ["LIMIT_BAL", "AGE", "PAY_0", "BILL_AMT1"]})
print("true label:", y_test[i], "| notebook P(default):", round(p_default, 6))

**D2-25 · PyTorch note: state_dict (run on Kaggle)**

In [ ]:
import torch
net = torch.nn.Sequential(torch.nn.Linear(23, 16), torch.nn.ReLU(), torch.nn.Linear(16, 2))
torch.save(net.state_dict(), "credit_mlp.pt")          # save: only the weights
net.load_state_dict(torch.load("credit_mlp.pt"))      # load into the same architecture
print(list(net.state_dict().keys()))